# Limpeza de Dados - ENEM (Microdados)

Este notebook lê os arquivos CSV brutos gigantes do ENEM (2013-2022), aplica os filtros geográficos para a Baixada Fluminense, mapeia as principais variáveis socioeconômicas usando os dicionários do INEP e salva os resultados em arquivos Parquet super otimizados mantendo a granularidade de microdados (1 linha = 1 participante).

In [1]:
import polars as pl
import os
import glob
import gc

In [2]:
# Lista de Municípios da Baixada Fluminense
BAIXADA_MUNICIPIOS = [
    'BELFORD ROXO', 'DUQUE DE CAXIAS', 'GUAPIMIRIM', 'ITAGUAI', 'ITAGUAÍ', 
    'JAPERI', 'MAGE', 'MAGÉ', 'MESQUITA', 'NILOPOLIS', 'NILÓPOLIS', 
    'NOVA IGUACU', 'NOVA IGUAÇU', 'PARACAMBI', 'QUEIMADOS', 
    'SAO JOAO DE MERITI', 'SÃO JOÃO DE MERITI', 'SEROPEDICA', 'SEROPÉDICA'
]

# Dicionários Universais (mantidos ao longo de 2013 a 2022)
DICT_RENDA = {
    'A': 'Nenhuma renda',
    'B': 'Até 1 SM',
    'C': 'De 1 a 1,5 SM',
    'D': 'De 1,5 a 2 SM',
    'E': 'De 2 a 2,5 SM',
    'F': 'De 2,5 a 3 SM',
    'G': 'De 3 a 4 SM',
    'H': 'De 4 a 5 SM',
    'I': 'De 5 a 6 SM',
    'J': 'De 6 a 7 SM',
    'K': 'De 7 a 8 SM',
    'L': 'De 8 a 9 SM',
    'M': 'De 9 a 10 SM',
    'N': 'De 10 a 12 SM',
    'O': 'De 12 a 15 SM',
    'P': 'De 15 a 20 SM',
    'Q': 'Mais de 20 SM'
}

DICT_ESCOLARIDADE = {
    'A': 'Nunca estudou',
    'B': 'Fundamental Incompleto (Até 5º ano)',
    'C': 'Fundamental Incompleto (Até 9º ano)',
    'D': 'Médio Incompleto',
    'E': 'Médio Completo',
    'F': 'Superior Completo',
    'G': 'Pós-graduação',
    'H': 'Não sei'
}

DICT_RACA = {
    0: 'Não declarado',
    1: 'Branca',
    2: 'Preta',
    3: 'Parda',
    4: 'Amarela',
    5: 'Indígena',
    6: 'Não disp'
}

DICT_ESCOLA = {
    1: 'Não Respondeu',
    2: 'Pública',
    3: 'Privada',
    4: 'Exterior'
}

In [ ]:
raw_dir = '../raw_data/microdados-enem'
out_dir = '../curated/parquet/enem/microdados_por_ano'
os.makedirs(out_dir, exist_ok=True)

anos_processados = []

for ano in range(2013, 2023):
    print(f"\nIniciando processamento do ano {ano}...")
    
    # Procura o arquivo CSV do ENEM na subpasta DADOS
    pattern = f"{raw_dir}/microdados_enem_{ano}/DADOS/*.csv"
    files = glob.glob(pattern)
    files = [f for f in files if 'MICRODADOS' in os.path.basename(f).upper() and 'ITENS' not in os.path.basename(f).upper()]
    
    if not files:
        # Tenta arquivos .txt, se existir
        pattern = f"{raw_dir}/microdados_enem_{ano}/DADOS/*.txt"
        files = glob.glob(pattern)
        files = [f for f in files if 'MICRODADOS' in os.path.basename(f).upper() and 'ITENS' not in os.path.basename(f).upper()]
    
    if not files:
        print(f"  Arquivo bruto não encontrado para {ano}!")
        continue
        
    file_path = files[0]
    print(f"  Lendo {file_path}...")
    
    try:
        # Lê o CSV com Polars
        # O principal delimitador é o ponto e vírgula
        # Lê para descobrir as colunas que existem
        df_head = pl.read_csv(file_path, separator=';', n_rows=1, encoding='iso-8859-1', ignore_errors=True)
        exist_cols = df_head.columns
        
        # Colunas que queremos extrair (se existirem no arquivo)
        target_cols = ['NU_ANO', 'CO_MUNICIPIO_ESC', 'NO_MUNICIPIO_ESC', 'CO_MUNICIPIO_PROVA', 'NO_MUNICIPIO_PROVA', 
                       'CO_MUNICIPIO_RESIDENCIA', 'NO_MUNICIPIO_RESIDENCIA', 'TP_DEPENDENCIA_ADM_ESC', 'TP_ESCOLA', 
                       'IN_TREINEIRO', 'TP_ST_CONCLUSAO', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT',
                       'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO',
                       'Q001', 'Q002', 'Q006', 'TP_COR_RACA']
                       
        keep_cols = [c for c in target_cols if c in exist_cols]
        
        # Lê o CSV 
        df_collected = pl.read_csv(file_path, separator=';', encoding='iso-8859-1', columns=keep_cols, infer_schema_length=10000, ignore_errors=True)
        
        # Transforma num LazyFrame
        df = df_collected.lazy()
    except Exception as e:
        print(f"  Erro ao ler: {e}")
        continue
        
    cols = df.columns
    
    # 1. Filtro Geográfico da Baixada Fluminense
    exprs = []
    for c in ['NO_MUNICIPIO_RESIDENCIA', 'NO_MUNICIPIO_PROVA', 'NO_MUNICIPIO_ESC']:
        if c in cols:
            exprs.append(pl.col(c).cast(pl.Utf8).str.to_uppercase().str.strip_chars().is_in(BAIXADA_MUNICIPIOS))
            
    if not exprs:
        print(f"  Nenhuma coluna de município encontrada em {ano}!")
        continue
        
    filter_expr = exprs[0]
    for e in exprs[1:]:
        filter_expr = filter_expr | e
        
    df = df.filter(filter_expr)
    
    # 2. Remoção de Treineiros e Ausentes
    # Treineiro
    if 'IN_TREINEIRO' in cols:
        df = df.filter(pl.col('IN_TREINEIRO') != 1)
    elif 'TP_ST_CONCLUSAO' in cols:
        df = df.filter(pl.col('TP_ST_CONCLUSAO') != 3)
        
    # Presença (0=Faltou, 1=Presente, 2=Eliminado)
    pres_cols = [c for c in cols if 'TP_PRESENCA' in c]
    for pc in pres_cols:
        df = df.filter(pl.col(pc) == 1)
        
    # 3. Engenharia de Features: Nota Média
    nota_cols = [c for c in cols if 'NU_NOTA' in c and 'REDACAO' not in c]
    if len(nota_cols) == 4:
        nota_exprs = [pl.col(c).cast(pl.Float64) for c in nota_cols]
        media_expr = (nota_exprs[0] + nota_exprs[1] + nota_exprs[2] + nota_exprs[3]) / 4.0
        df = df.with_columns(media_expr.alias('NOTA_MEDIA_OBJ'))
        
    # 4. Mapeamento Socioeconômico e Demográfico
    # Renda (Q006)
    if 'Q006' in cols:
        df = df.with_columns(pl.col('Q006').cast(pl.Utf8).str.to_uppercase().replace_strict(DICT_RENDA, default=pl.col('Q006')).alias('RENDA_FAMILIAR'))
    # Mãe (Q002)
    if 'Q002' in cols:
        df = df.with_columns(pl.col('Q002').cast(pl.Utf8).str.to_uppercase().replace_strict(DICT_ESCOLARIDADE, default=pl.col('Q002')).alias('ESCOLARIDADE_MAE'))
    # Pai (Q001)
    if 'Q001' in cols:
        df = df.with_columns(pl.col('Q001').cast(pl.Utf8).str.to_uppercase().replace_strict(DICT_ESCOLARIDADE, default=pl.col('Q001')).alias('ESCOLARIDADE_PAI'))
    # Raça (TP_COR_RACA)
    if 'TP_COR_RACA' in cols:
        df = df.with_columns(pl.col('TP_COR_RACA').cast(pl.Int32, strict=False).replace_strict(DICT_RACA, default=pl.col('TP_COR_RACA').cast(pl.Utf8)).alias('RACA_DESC'))
    # Escola (TP_ESCOLA)
    if 'TP_ESCOLA' in cols:
        df = df.with_columns(pl.col('TP_ESCOLA').cast(pl.Int32, strict=False).replace_strict(DICT_ESCOLA, default=pl.col('TP_ESCOLA').cast(pl.Utf8)).alias('TIPO_ESCOLA'))
        
    # Garante a coluna ANO
    if 'NU_ANO' not in cols:
        df = df.with_columns(pl.lit(ano).alias('NU_ANO'))
        
    print(f"  Extraindo dados para memória...")
    try:
        df_collected = df.collect()
    except Exception as e:
        print(f"  Erro ao processar (pode ser problema de aspas no CSV): {e}")
        # Tenta ler novamente desativando quote_char se houver linhas corrompidas
        df_fallback = pl.read_csv(file_path, separator=';', columns=keep_cols, infer_schema_length=10000, ignore_errors=True, encoding='iso-8859-1', quote_char=None).lazy()
        df_fallback = df_fallback.filter(filter_expr)
        df_collected = df_fallback.collect()
        
    linhas = df_collected.height
    
    out_file = f"{out_dir}/enem_microdados_{ano}.parquet"
    df_collected.write_parquet(out_file)
    print(f"  Ano {ano} concluído: {linhas} registros salvos em {out_file}")
    
    anos_processados.append(out_file)
    
    # Libera memória
    del df_collected
    gc.collect()



Iniciando processamento do ano 2013...
  Lendo ../raw_data/microdados-enem/microdados_enem_2013/DADOS/MICRODADOS_ENEM_2013.csv...


/var/folders/mv/14j2pr4166n3rk5pm57jds_r0000gn/T/ipykernel_7508/294735914.py:53: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  cols = df.columns


  Extraindo dados para memória...
  Ano 2013 concluído: 69593 registros salvos em ../curated/parquet/enem/microdados_por_ano/enem_microdados_2013.parquet

Iniciando processamento do ano 2014...
  Lendo ../raw_data/microdados-enem/microdados_enem_2014/DADOS/MICRODADOS_ENEM_2014.csv...
  Extraindo dados para memória...
  Ano 2014 concluído: 85826 registros salvos em ../curated/parquet/enem/microdados_por_ano/enem_microdados_2014.parquet

Iniciando processamento do ano 2015...
  Lendo ../raw_data/microdados-enem/microdados_enem_2015/DADOS/MICRODADOS_ENEM_2015.csv...
  Extraindo dados para memória...
  Ano 2015 concluído: 85179 registros salvos em ../curated/parquet/enem/microdados_por_ano/enem_microdados_2015.parquet

Iniciando processamento do ano 2016...
  Lendo ../raw_data/microdados-enem/microdados_enem_2016/DADOS/microdados_enem_2016.csv...
  Extraindo dados para memória...
  Ano 2016 concluído: 76601 registros salvos em ../curated/parquet/enem/microdados_por_ano/enem_microdados_2016

In [4]:
# Consolidação
print("\nIniciando consolidação de todos os anos...")
dfs = []
for file in anos_processados:
    # Para consolidar, lemos apenas as colunas em comum mais as features calculadas
    df = pl.read_parquet(file)
    dfs.append(df)
    
# Como os esquemas podem variar ao longo dos anos (ex: Q070 existia em 2013 mas não em 2022),
# o concat 'diagonal' (how='diagonal_relaxed') preenche com Nulos as colunas faltantes!
print("Concatenando dados (Diagonal Relaxed)...")
df_final = pl.concat(dfs, how='diagonal_relaxed')

final_path = '../curated/parquet/enem/dataset_enem_microdados_baixada.parquet'
df_final.write_parquet(final_path)

print(f"Consolidação concluída!")
print(f"Total de registros na base final: {df_final.height}")
print(f"Salvo em: {final_path}")



Iniciando consolidação de todos os anos...
Concatenando dados (Diagonal Relaxed)...
Consolidação concluída!
Total de registros na base final: 467866
Salvo em: ../curated/parquet/enem/dataset_enem_microdados_baixada.parquet
